In [1]:
import sys

sys.path.insert(0, "/Users/shelleygoel/Code/01_statistical_mod_blog/anomaly_detection")

import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from core.dataset import TimeSeriesDataset
from core.evaluation import Evaluation
from core.feature_transformer import C22Feature, FeatureCategory, FeatureTransformer
from core.hvac_data_gen import HVACDataGenerator
from core.models import Catch22MPModel, EuclideanDistModel, FeatureWeighter, UniformWeighter, IForestModel
from core.viz import plot_cases

/Users/shelleygoel/miniconda3/envs/TSB-AD/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load HVAC Data

In [ ]:
skip_data_gen = True
if not skip_data_gen:
    # Generate HVAC Data with longer history
    generator = HVACDataGenerator(seed=10)
    # anomaly_config = [
    #         {
    #             'unit': 1,
    #             'type': 'frequency',
    #             'start_day': 0,
    #             'start_hour': 8,
    #             'duration_hours': 24*5,
    #             'params': {'frequency_multiplier': 0.2}
    #         },
    #     ]
        
    hvac_df = generator.generate_dataset(
        num_containers=500,
        start_time=datetime(2026, 1, 15),
        duration_days=20,
    )
else:
    hvac_df = pd.read_parquet(Path("../datasets/hvac_anomalies_v021226.parquet"))

    hvac_df = hvac_df.copy()

# Post processing: smooth TmpRet    
hvac_df["TmpRet"] = hvac_df.groupby(["container_id", "unit"])["TmpRet"].transform(
        lambda x: x.rolling(window=10, min_periods=1).mean()
    )

# Wrap in TimeSeriesDataset
col_map = {
    "entity": "container_id",
    "time": "timestamp_et",
    "value_cols": ["TmpRet"],
    "label": "anomaly",
    "label_type": "anomaly_type",
    "sub_entity": "unit",
}
hvac_ds = TimeSeriesDataset(hvac_df, col_map)
print(hvac_ds.anomaly_summary())

sample_for_exp = False
if sample_for_exp:
    n_sample_size = 100
    sampled_entities = hvac_ds.sample_entities(n_cases=50, label_type="frequency", random_state=42)
    normal_entities = hvac_ds.sample_entities(n_cases=n_sample_size, label_type="normal", random_state=42)

    train_entities = np.concatenate([sampled_entities, normal_entities])
    hvac_ds = TimeSeriesDataset(hvac_df[hvac_df["container_id"].isin(train_entities)], col_map)
    print(hvac_ds.anomaly_summary())

  label_type  entity_count
0     normal           911
1        lag            38
2  amplitude            27
3  frequency            24
  label_type  entity_count
0     normal           911
1        lag            38
2  amplitude            27
3  frequency            24



# Euclidean Distance Model


In [3]:

eucl = EuclideanDistModel(feature_col="TmpRet", smooth_window=1, dist_window=12 * 60, strategy="iqr")
eucl_day_scores = eucl.score_anomalies(hvac_ds, level="day")

# Catch22 MP Model

## C22 Features Calculation

In [4]:
# 3. Feature transform
feats_to_calc = [
    C22Feature.CO_f1ecac,
    C22Feature.CO_FirstMin_ac,
    C22Feature.IN_AutoMutualInfoStats_40_gaussian_fmmi,
    C22Feature.SP_Summaries_welch_rect_area_5_1,
    C22Feature.SP_Summaries_welch_rect_centroid,
]
ft = FeatureTransformer(
    raw_data_columns=["TmpRet"], window_size=12 * 60, stride=60, n_jobs=8, c22_features=feats_to_calc
)
feat_ds = ft.transform(hvac_ds)  # categories=[FeatureCategory.C22_RAW_DIFF])

FeatureTransformer: 100%|██████████| 1000/1000 [00:53<00:00, 18.58it/s]


## C22MP: Score Anomalies
- Left C22MP Profile Calculation
- day Level Scores

In [5]:
calc_features = sum([list(v) for v in ft.feature_map.values()], [])


class CustomWeighter(FeatureWeighter):
    def compute_weights(self, preferred_features: list) -> dict:
        return dict([(feat, 1) for feat in preferred_features])


pattern = re.compile(
    r"(?=.*_featdiff)(?=.*(SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid|PD_PeriodicityWang_th0.01|CO_f1ecac|CO_FirstMin_ac|IN_AutoMutualInfoStats_40_gaussian_fmmi))"
)
pattern = re.compile(
    r"(?=.*_featdiff)(?=.*(SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid|CO_f1ecac))"
)
pattern = re.compile(
    r"^(?!.*_featdiff).*(SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid|CO_f1ecac)"
)
# pattern = re.compile(r'(?=.*_featdiff)(?=.*(CO_f1ecac))')
pattern = re.compile(r"^(?!.*_featdiff).*(0_1__CO_f1ecac|0_2__CO_f1ecac|1_2__CO_f1ecac)")
# pattern = re.compile(r'SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid')

pref_features = [col for col in calc_features if pattern.search(col)] + [
    "TmpRet__featdiff_0_1__CO_f1ecac",
    "TmpRet__featdiff_0_2__CO_f1ecac",
    "TmpRet__featdiff_1_2__CO_f1ecac",
]
cw = CustomWeighter()
pref_feat_exp = [
    ("raw_feat", ["TmpRet__0__CO_f1ecac", "TmpRet__1__CO_f1ecac", "TmpRet__2__CO_f1ecac"]),
    (
        "feat_diff",
        [
            "TmpRet__featdiff_0_1__CO_f1ecac",
            "TmpRet__featdiff_0_2__CO_f1ecac",
            "TmpRet__featdiff_1_2__CO_f1ecac",
        ],
    ),
    ("raw_diff", ["TmpRet__0_1__CO_f1ecac", "TmpRet__0_2__CO_f1ecac", "TmpRet__1_2__CO_f1ecac"]),
]
stride = ft.stride  # 45
exclude_zone = 1440 // stride  # 32
print(f"{stride=}, {exclude_zone=}")
c22_mps = {}
c22_day_scores = {}
for model_name, pref_features in pref_feat_exp:
    custom_wei = cw.compute_weights(preferred_features=pref_features)
    # 4. Fit MP once
    c22mp = Catch22MPModel(
        exclude_zone=exclude_zone,
        early_abandon=False,  # brute force — fast enough
    )
    mp_ds, debug = c22mp.fit_profile(feat_ds, weights=custom_wei)
    scores = c22mp.score(mp_ds, level="day", day_agg_stat="p90")
    c22_mps[model_name] = mp_ds
    c22_day_scores[model_name] = scores


stride=60, exclude_zone=24


Catch22MP fit_profile: 100%|██████████| 1000/1000 [00:00<00:00, 1713.50it/s]


In [16]:
px.histogram(c22_day_scores["feat_diff"].df["anomaly_score"])

# Isolation Forest Model

In [27]:
iforest = IForestModel(fit_scope="global", contamination=0.02, feature_cols=pref_feat_exp[1][1])
iforest_day_scores = iforest.score_anomalies(feat_ds, level="day", day_agg_stat="p90")
px.histogram(iforest_day_scores.df["anomaly_score"])

# Compare Models: PR curves

In [28]:
ev = Evaluation(level="day")
fig = ev.plot_pr_curves_compared(
    # {"Catch22MP": c22_day_scores},
    {**c22_day_scores, "eucl_dist": eucl_day_scores, "iforest": iforest_day_scores},
    hvac_ds,
)
fig.show()

# Appendix

## Feat Selection

In [ ]:
from sklearn.model_selection import train_test_split

# Entity lists (already identified upstream)
abnormal_entities = sampled_entities  # the 50 "frequency" cases
normal_entities_arr = normal_entities  # the 400 normals

# Stratified split on entities — preserves anomaly ratio in both halves
ab_train, ab_test = train_test_split(abnormal_entities, test_size=0.6, random_state=42)
no_train, no_test = train_test_split(normal_entities_arr, test_size=0.6, random_state=42)

train_entities = np.concatenate([ab_train, no_train])
test_entities = np.concatenate([ab_test, no_test])

ds_train = TimeSeriesDataset(hvac_df[hvac_df["container_id"].isin(train_entities)], col_map)
ds_test = TimeSeriesDataset(hvac_df[hvac_df["container_id"].isin(test_entities)], col_map)


def ratio(name, ab, no):
    n = len(ab) + len(no)
    print(f"{name}: n_entities={n}, anomaly={len(ab)}, normal={len(no)}, ratio={len(ab) / n:.3f}")


ratio("train", ab_train, no_train)
ratio("test", ab_test, no_test)

In [ ]:
entity_col = feat_ds.col_map["entity"]

feat_ds_train = TimeSeriesDataset(
    feat_ds.df[feat_ds.df[entity_col].isin(train_entities)].copy(),
    feat_ds.col_map,
)
feat_ds_test = TimeSeriesDataset(
    feat_ds.df[feat_ds.df[entity_col].isin(test_entities)].copy(),
    feat_ds.col_map,
)

print(f"feat_ds_train: {feat_ds_train.df[entity_col].nunique()} entities, {len(feat_ds_train.df)} rows")
print(f"feat_ds_test:  {feat_ds_test.df[entity_col].nunique()} entities, {len(feat_ds_test.df)} rows")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# --- 1. Label each feature window ---
# A window (starting at feat_ds timestamp t) is anomalous if ANY raw timestamp
# in [t, t + window_size) has label==1.
window_size = ft.window_size
entity_col = ds_small.col_map["entity"]
time_col = ds_small.col_map["time"]
label_col = ds_small.col_map["label"]

ts_labels_df = ds_train.ts_labels()  # entity, time, label, [label_type]

window_label_rows = []
for eid, feat_grp in feat_ds_train.df.groupby(entity_col):
    raw = (
        ts_labels_df[ts_labels_df[entity_col] == eid].set_index(time_col)[label_col].sort_index().astype(int)
    )
    # Rolling max is right-anchored (covers [t-W+1, t]); shift by -(W-1) so the
    # value at t covers [t, t+W-1] — i.e. a window STARTING at t.
    fwd_max = raw.rolling(window=window_size, min_periods=1).max().shift(-(window_size - 1))

    tmp = feat_grp[[entity_col, time_col]].copy()
    tmp["window_label"] = fwd_max.reindex(feat_grp[time_col].values).values
    window_label_rows.append(tmp)

window_labels_df = pd.concat(window_label_rows, ignore_index=True)

# --- 2. Build (X, y) ---
merged = feat_ds_train.df.merge(window_labels_df, on=[entity_col, time_col])
merged = merged.dropna(subset=["window_label"])  # drop right-edge windows that overflow

feature_cols = feat_ds_train.col_map["value_cols"]
X = merged[feature_cols].values
y = merged["window_label"].astype(int).values

print(f"n_windows={len(y)}, n_anomaly_windows={y.sum()}, base_rate={y.mean():.3f}")

# --- 3. Fit Decision Tree + rank features ---
clf = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",  # anomalies are rare — rebalance
    random_state=42,
)
clf.fit(X, y)

feat_imp = (
    pd.DataFrame({"feature": feature_cols, "importance": clf.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
print(feat_imp.head(20))

### Features for frequency
Direct spectral (Welch power spectrum)
  - SP_Summaries_welch_rect_area_5_1 — power in the lowest 1/5 of the spectrum. Shifts when dominant frequency moves.
  - SP_Summaries_welch_rect_centroid — frequency at which power is concentrated. Clearest signal for "wrong period."

  Periodicity detectors
  - PD_PeriodicityWang_th0.01 — Wang's dominant-periodicity estimate. Directly encodes the cycle length.
  - CO_f1ecac — first crossing of ACF at 1/e. A characteristic timescale — changes with period.
  - CO_FirstMin_ac — first minimum of the autocorrelation function. Roughly half the dominant period.
  - IN_AutoMutualInfoStats_40_gaussian_fmmi — first minimum of Auto-Mutual-Information. Nonlinear analog of CO_FirstMin_ac.

  Scaling / long-range (weaker, indirect)
  - SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1 — DFA scaling exponent.
  - SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1 — R/S range fit. Both reflect how power distributes across scales.

  Residual-autocorrelation
  - FC_LocalSimple_mean1_tauresrat — ratio of residual ACF timescale to raw ACF timescale after a simple forecast. Sensitive to how "learnable" the periodic structure is.

In [ ]:
ev.auc_pr(scores=c22_day_scores["feat_diff"], dataset=ds_small)

In [ ]:
feat_cfg = feat_ds.to_plot_cfg()
feat_cfg.value_cols = (
    # pref_features  # sorted([col for col in feat_ds.to_plot_cfg().value_cols if col.find("0_2") != -1] )
    pref_feat_exp[1][1]
)
skip = False
lbt = "frequency"
# lbt = "normal"
if not skip:
    figs = plot_cases(
        # [ds_small.to_plot_cfg(),
        [ds_small.to_plot_cfg(), mp_ds.to_plot_cfg(), feat_cfg],
        sample_from=ds_small,
        n_cases=10,
        # entity_ids=[246],
        # label_type="frequency",
        label_type=lbt,
        labels_from=ds_small.to_plot_cfg(),
        random_state=42,
    )


ckpt_dir = Path(f"checkpoints/hvac/{lbt}")
ckpt_dir.mkdir(exist_ok=True)

for i, fig in enumerate(figs):
    fig.write_html(ckpt_dir / f"case_{i}.html")
    # fig.write_image(ckpt_dir / f"case_{i}.png", width=1200, height=600, scale=2)
    # fig.show()

# 7. Evaluate
# ev = Evaluation(level='day')
# print(ev.metrics_table(day_scores, ds_small))

In [ ]:
eucl = EuclideanDistModel(feature_col="TmpRet", smooth_window=1, dist_window=6 * 60, strategy="mad")
eucl_day_scores = eucl.score_anomalies(ds_small, level="day")

# Catch22 MP scoring (already have day_scores from earlier)
# day_scores = c22mp.score(mp_ds, level="day")


In [ ]:
# Compare
ev = Evaluation(level="day")

# Metrics side-by-side
print(
    ev.compare(
        {"Catch22MP": c22_day_scores, "EuclideanDist": eucl_day_scores},
        ds_small,
    )
)

# Overlaid PR curves, one subplot per anomaly type
fig = ev.plot_pr_curves_compared(
    {"Catch22MP": c22_day_scores, "EuclideanDist": eucl_day_scores},
    ds_small,
)
fig.show()